In [2]:
import os 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_log_error
os.chdir("../")
os.getcwd()

'/Users/pierrequintindekercadio/Desktop/MOSEF/S1/projet_time_series/Store_Sales_Time_Series_Forecasting'

In [3]:
sales_df = pd.read_csv('data/sales_df.csv')
sales_df['date'] = pd.to_datetime(sales_df['date'])


In [4]:
sales_df.columns

Index(['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city',
       'state', 'store_type', 'cluster', 'dcoilwtico', 'holiday_type',
       'is_holiday_local', 'is_holiday_regional', 'is_holiday_national',
       'is_store_closed', 'day_of_week', 'month', 'year', 'time_idx',
       'is_weekend', 'is_payday', 'month_sin', 'month_cos', 'day_sin',
       'day_cos', 'is_earthquake_impact', 'days_to_christmas', 'sales_lag_1',
       'sales_lag_2', 'sales_lag_3', 'sales_lag_4', 'sales_lag_5',
       'sales_lag_6', 'sales_lag_7', 'sales_lag_14', 'sales_lag_21',
       'sales_lag_28', 'sales_lag_56', 'sales_lag_364', 'rolling_mean_7',
       'rolling_mean_28', 'rolling_mean_56', 'rolling_std_7',
       'store_family_velocity', 'target_enc_store_family', 'oil_trend_30',
       'promo_ratio_vs_avg', 'promo_during_payday'],
      dtype='object')

# 1 - Expanding Window à 8 semaines

In [5]:
def rmsle(y_true, y_pred):
    return np.sqrt(mean_squared_log_error(y_true, np.maximum(0, y_pred)))


def evaluate_expanding_window(serie, model_func, n_splits=3, test_size=56):
    tscv = TimeSeriesSplit(n_splits=n_splits, test_size=test_size)
    scores = []
    
    for fold, (train_idx, val_idx) in enumerate(tscv.split(serie)):
        train_data, val_data = serie[train_idx], serie[val_idx]
    
        preds = model_func(train_data, len(val_data))
        
        score = rmsle(val_data, preds)
        scores.append(score)
        print(f"  Fold {fold+1}: RMSLE = {score:.4f}")
        
    return np.mean(scores)


# 2 - Modèle ARMA

In [10]:
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from tqdm import tqdm  # Pour voir la progression sans polluer la console

def run_arma(train_data, forecast_steps):
    model = SARIMAX(train_data, order=(1, 0, 1), 
                    enforce_stationarity=False, 
                    enforce_invertibility=False)
    results = model.fit(disp=False)
    return results.forecast(steps=forecast_steps)

all_results = []
grouped = sales_df.groupby(['store_nbr', 'family'])

# On utilise tqdm pour suivre l'avancée sur les ~1700 combinaisons
print(f"⏳ Calcul en cours pour tous les magasins et familles...")

for (store, fam), data in tqdm(grouped):
    daily_data = data.sort_values('date')
    serie_val = daily_data['sales'].fillna(0).values
    
    if len(serie_val) > 200:
        try:
            avg_score = evaluate_expanding_window(serie_val, run_arma)
            all_results.append({
                'store': store,
                'family': fam,
                'rmse_score': avg_score
            })
        except:
            continue

# --- AFFICHAGE DU TABLEAU FINAL UNIQUEMENT ---
results_df = pd.DataFrame(all_results)
summary_pivot = results_df.pivot(index='family', columns='store', values='rmse_score')

print("\n" + "="*30)
print("SYNTHÈSE FINALE DES SCORES RMSE")
print("="*30)
# On affiche tout le tableau (ou une partie si c'est trop large)
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(summary_pivot)

# Sauvegarde automatique pour analyse externe
results_df.to_csv("synthese_ventes_arma.csv")

⏳ Calcul en cours pour tous les magasins et familles...


  0%|          | 0/1782 [00:00<?, ?it/s]

  Fold 1: RMSLE = 0.5387
  Fold 2: RMSLE = 0.5842


  0%|          | 1/1782 [00:01<32:21,  1.09s/it]/Users/pierrequintindekercadio/Desktop/MOSEF/S1/projet_time_series/Store_Sales_Time_Series_Forecasting/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/pierrequintindekercadio/Desktop/MOSEF/S1/projet_time_series/Store_Sales_Time_Series_Forecasting/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/pierrequintindekercadio/Desktop/MOSEF/S1/projet_time_series/Store_Sales_Time_Series_Forecasting/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization fail

  Fold 3: RMSLE = 0.6138
  Fold 1: RMSLE = 0.0000
  Fold 2: RMSLE = 0.0000
  Fold 3: RMSLE = 0.0000


...

  Fold 3: RMSLE = 0.3185
  Fold 1: RMSLE = 0.2501
  Fold 2: RMSLE = 0.2656


100%|█████████▉| 1781/1782 [05:39<00:00,  5.56it/s]

  Fold 3: RMSLE = 0.2483
  Fold 1: RMSLE = 1.9105
  Fold 2: RMSLE = 3.0290
  Fold 3: RMSLE = 0.0000
  Fold 1: RMSLE = 0.6573
  Fold 2: RMSLE = 0.7713


100%|██████████| 1782/1782 [05:39<00:00,  5.25it/s]

  Fold 3: RMSLE = 0.7429

SYNTHÈSE FINALE DES SCORES RMSE
store                             1         2         3         4         5   \
family                                                                         
AUTOMOTIVE                  0.578907  0.621807  0.459290  0.575659  0.452990   
BABY CARE                   0.000000  0.307402  0.621125  0.117618  0.311400   
BEAUTY                      0.548515  0.552635  0.517454  0.620519  0.498056   
BEVERAGES                   0.366417  0.245679  0.551937  0.293681  0.240159   
BOOKS                       0.404961  0.138161  0.333613  0.326981  0.221428   
BREAD/BAKERY                0.372809  0.192832  0.209568  0.256467  0.160707   
CELEBRATION                 0.719848  0.528026  0.553958  0.789363  0.420543   
CLEANING                    0.452183  0.200845  0.215683  0.233199  0.207378   
DAIRY                       0.341489  0.236273  0.228442  0.270425  0.158536   
DELI                        0.366145  0.236247  1.451865  0.28

1700 

fais les familles avec les magasins

## 3 - SARIMAX

In [11]:
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm

# --- 1. Définition du moteur SARIMAX ---
def run_sarimax_exog(train_data, train_exog, forecast_steps, test_exog):
    model = SARIMAX(train_data, 
                    exog=train_exog,
                    order=(1, 1, 1), 
                    seasonal_order=(1, 1, 1, 7),
                    enforce_stationarity=False,
                    enforce_invertibility=False)
    
    results = model.fit(disp=False)
    forecast = results.forecast(steps=forecast_steps, exog=test_exog)
    return np.maximum(0, forecast)

# --- 2. Configuration ---
exog_cols = [
    'onpromotion', 'dcoilwtico', 'is_payday', 
    'is_holiday_national', 'is_weekend', 
    'month_sin', 'month_cos', 'day_sin', 'day_cos'      
]

all_results_sarimax = []
grouped = sales_df.groupby(['store_nbr', 'family'])

print(f"⏳ Lancement SARIMAX (Saisonnalité 7j + Exogènes) sur TOUS les magasins...")

# --- 3. Boucle Globale ---
# On utilise tqdm pour voir l'avancement global
for (store, fam), df_group in tqdm(grouped):
    df_group = df_group.sort_values('date')
    
    serie_y = df_group['sales'].fillna(0).values
    serie_x = df_group[exog_cols].fillna(0).values
    
    # Sécurité : On a besoin d'assez de données pour la saisonnalité (au moins quelques semaines)
    if len(serie_y) > 300: 
        try:
            tscv = TimeSeriesSplit(n_splits=3, test_size=56)
            fold_scores = []
            
            for train_idx, val_idx in tscv.split(serie_y):
                y_train, y_val = serie_y[train_idx], serie_y[val_idx]
                x_train, x_val = serie_x[train_idx], serie_x[val_idx]
                
                preds = run_sarimax_exog(y_train, x_train, len(y_val), x_val)
                
                # On utilise ta fonction rmsle
                score = rmsle(y_val, preds)
                fold_scores.append(score)
            
            all_results_sarimax.append({
                'store': store,
                'family': fam,
                'mean_rmsle': np.mean(fold_scores)
            })
            
        except Exception:
            # En cas de non-convergence du modèle sur un groupe spécifique
            continue

# --- 4. Affichage du Tableau de Synthèse Final ---
results_df = pd.DataFrame(all_results_sarimax)
summary_pivot = results_df.pivot(index='family', columns='store', values='mean_rmsle')

print("\n" + "="*40)
print("SYNTHÈSE FINALE SARIMAX + EXOGÈNES (RMSLE)")
print("="*40)

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(summary_pivot)

# Sauvegarde pour ne pas perdre le calcul
results_df.to_csv("synthese_globale_sarimax_exog.csv", index=False)

⏳ Lancement SARIMAX (Saisonnalité 7j + Exogènes) sur TOUS les magasins...


  0%|          | 0/1782 [00:00<?, ?it/s]/Users/pierrequintindekercadio/Desktop/MOSEF/S1/projet_time_series/Store_Sales_Time_Series_Forecasting/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/pierrequintindekercadio/Desktop/MOSEF/S1/projet_time_series/Store_Sales_Time_Series_Forecasting/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/pierrequintindekercadio/Desktop/MOSEF/S1/projet_time_series/Store_Sales_Time_Series_Forecasting/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



SYNTHÈSE FINALE SARIMAX + EXOGÈNES (RMSLE)
store                             1         2         3         4         5   \
family                                                                         
AUTOMOTIVE                  0.567780  0.586192  0.399191  0.541832  0.457945   
BABY CARE                   0.000013  0.318681  0.614835  0.155148  0.313486   
BEAUTY                      0.499965  0.472409  0.408431  0.509933  0.451724   
BEVERAGES                   0.877811  0.215959  0.172953  0.201571  0.196100   
BOOKS                       0.441416  0.225023  0.394139  0.400884  0.262183   
BREAD/BAKERY                0.666993  0.151224  0.156179  0.196535  0.146070   
CELEBRATION                 0.922908  0.493964  0.652624  0.629578  0.408309   
CLEANING                    0.235194  0.189177  0.190620  0.228568  0.262688   
DAIRY                       0.213766  0.211626  0.186829  0.192391  0.185980   
DELI                        0.251427  0.227439  0.216106  0.249677  0.192264

In [12]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX

# 1. Préparation des données spécifiques
magasin = 1
famille = 'GROCERY I'

# Filtrage
df_grocery = sales_df[(sales_df['store_nbr'] == magasin) & 
                      (sales_df['family'] == famille)].sort_values('date')

# Nettoyage
serie_val = df_grocery['sales'].fillna(0).values

print(f"📊 Analyse pour le Magasin {magasin} | Famille: {famille}")
print(f"Nombre de jours de données : {len(serie_val)}")

# 2. Définition et entraînement du modèle ARMA(1,0,1)
# Rappel : order=(p, d, q) -> d=0 pour un modèle ARMA pur
def calculate_single_arma(data):
    # On utilise les 80% premières données pour le train, 20% pour le test
    split = int(len(data) * 0.8)
    train, test = data[:split], data[split:]
    
    model = SARIMAX(train, 
                    order=(1, 0, 1), 
                    enforce_stationarity=False, 
                    enforce_invertibility=False)
    results = model.fit(disp=False)
    
    # Prédiction sur la taille du set de test
    forecast = results.forecast(steps=len(test))
    forecast = np.maximum(0, forecast) # Pas de ventes négatives
    
    # Calcul du score RMSLE
    # rmsle = sqrt(mean((log(pred+1) - log(actual+1))^2))
    score = np.sqrt(np.mean(np.square(np.log1p(forecast) - np.log1p(test))))
    return score, forecast, test

# 3. Exécution et affichage
if len(serie_val) > 100:
    score_grocery, preds, actuals = calculate_single_arma(serie_val)
    print(f"\n✅ Calcul terminé")
    print(f"RMSLE pour GROCERY I (Magasin 1) : {score_grocery:.4f}")
else:
    print("❌ Pas assez de données pour cette combinaison.")

# Optionnel : Petit aperçu comparatif
comparaison = pd.DataFrame({'Réel': actuals[:5], 'Prédit': preds[:5]})
print("\n--- Aperçu des 5 premières prédictions ---")
print(comparaison)

📊 Analyse pour le Magasin 1 | Famille: GROCERY I
Nombre de jours de données : 1684

✅ Calcul terminé
RMSLE pour GROCERY I (Magasin 1) : 0.5642

--- Aperçu des 5 premières prédictions ---
     Réel       Prédit
0  2664.0  2320.217449
1  2623.0  2320.557198
2  2597.0  2320.896997
3  2290.0  2321.236845
4  2625.0  2321.576743
